In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    row_number,
    sha2,
    trim,
    when,
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_payment_schedule_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PaymentScheduleId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp,  BatchId string, PaymentScheduleKey string,"
        #     "ExposureKey String, PartyKey string, AgreementKey string, SettlementKey String, PaymentScheduleTypeCode string, PaymentScheduleType string,"
        #     "InstallmentNumber int, InstallmentDate timestamp, RequestSN int, InstallmentTypeCode string,"
        #     "InstallmentType string, InstallmentStatusCode string, InstallmentStatus string,"
        #     "InstallmentAmount decimal(38,7), InterestAmount decimal(19,2), CapitalAmount decimal(19,2),"
        #     "CommissionAmount decimal(19,2), ExpenseAmount decimal(19,2),"
        #     "OverdueDays int, InstallmentAmountPaid decimal(19,2), "
        #     "InterestAmountPaid decimal(19,2), CapitalAmountPaid decimal(19,2),"
        #     "CommissionAmountPaid decimal(19,2), ExpenseAmountPaid decimal(19,2),"
        #     "RemainingAmount decimal(19,2), DownPaymentFlag boolean, RecordKey int,"
        #     "SettlementCaseRecordKey int, PayTolerance int, PayPercent decimal(12,6), PayPercentAmount decimal(18,2),"
        #     "ImplementationFlag boolean, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.PaymentSchedule"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "portfolio_id_qca"]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ============================== fs_arrangements_plan ====================

        fs_arrangements_plan_extra_filters = None

        if fs_arrangements_plan_extra_filters is not None:
            fs_arrangements_plan_extra_filters.append(date_filter)
            fs_arrangements_plan_final_filters = " AND ".join(
                fs_arrangements_plan_extra_filters
            )
        else:
            fs_arrangements_plan_final_filters = date_filter

        fs_arrangements_plan_selected_fields = [
            "seps_code",
            "setc_code",
            "sete_code",
            "pd_desc",
            "seps_installment_type_code",
            "inst_id",
            "seps_installment_amt",
            "seps_is_down_payment",
            "cust_code",
            "case_code",
            "businessdate",
        ]

        fs_arrangements_plan = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_arrangements_plan",
            fs_arrangements_plan_final_filters,
            fs_arrangements_plan_selected_fields,
        )

        fs_arrangements_plan = fs_arrangements_plan.distinct()

        # HashKeys
        fs_arrangements_plan_primary_key_fields = [
            fs_arrangements_plan.seps_code,
            fs_arrangements_plan.setc_code,
            fs_arrangements_plan.sete_code,
            lit("qca"),
        ]

        fs_arrangements_plan = fs_arrangements_plan.withColumn(
            "PaymentScheduleId",
            sha2(concat_ws(";", *fs_arrangements_plan_primary_key_fields), 256),
        ).withColumn(
            "PaymentScheduleKey", concat_ws(";", "seps_code", "setc_code", "sete_code")
        )

        # ================== fs_cases_snap ==================

        fs_cases_snap_extra_filters = None

        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter

        fs_cases_snap_selected_fields = ["cust_code", "case_code", "SPET_CODE", "CASE_SOURCE"]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "case_code",
            "cust_code",
            "CASE_ACCOUNT_FOR_SEARCH",
            "CSTM_TEXT_AD_20",
            "CASE_CONTRACT_ACCOUNT_KEY",
            "CASE_LEGAL_CONTRACT",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        # ============= Join =============

        # Conditions
        fs_arrangements_plan_fs_cases_snap_join_condition = (
            fs_arrangements_plan.case_code == fs_cases_snap.case_code)

        fs_arrangements_plan_dim_cases_static_join_condition = (
            fs_arrangements_plan.case_code == dim_cases_static.case_code
        )

        fs_cases_snap_portfolio_join_condition = (
            fs_cases_snap["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        # Join
        join_payment_schedule_df = (
            fs_arrangements_plan.join(
                fs_cases_snap,
                fs_arrangements_plan_fs_cases_snap_join_condition,
                "inner",
            )
            .join(
                dim_cases_static,
                fs_arrangements_plan_dim_cases_static_join_condition,
                "inner",
            )
            .join(
                portfolio,
                fs_cases_snap_portfolio_join_condition,
                "left",
            )
        )

        # Adding 'InstallmentStatus' and 'InstallmentType' columns
        join_payment_schedule_df = join_payment_schedule_df.withColumn(
            "InstallmentStatus",
            when(join_payment_schedule_df.inst_id == "0", lit("Active"))
            .when(join_payment_schedule_df.inst_id == "1", lit("Overdue"))
            .when(join_payment_schedule_df.inst_id == "2", lit("Matched"))
        ).withColumn(
            "InstallmentType",
            when(join_payment_schedule_df.seps_installment_type_code == "2081", lit("Instalment-Cash"))
            .when(join_payment_schedule_df.seps_installment_type_code == "3021", lit("Instalment-Ρύθμιση"))
            .otherwise(lit("N/A"))
        ).withColumn(
            "AgreementKey",
            when(
                dim_cases_static.CSTM_TEXT_AD_20.isNull() | (trim(dim_cases_static.CSTM_TEXT_AD_20) == ""),
                when(
                    (fs_cases_snap.CASE_SOURCE).like("C%"),
                    dim_cases_static.CASE_CONTRACT_ACCOUNT_KEY,
                ).otherwise(dim_cases_static.CASE_LEGAL_CONTRACT)
            ).otherwise(dim_cases_static.CSTM_TEXT_AD_20)
        ).withColumn(
            "InstallmentNumber",
            row_number().over(
                Window.partitionBy(fs_arrangements_plan.sete_code, dim_cases_static.CASE_ACCOUNT_FOR_SEARCH)
                .orderBy(fs_arrangements_plan.pd_desc.asc())
            )
        )

        # ============================== payment_schedule_qca ====================

        payment_schedule_qca = join_payment_schedule_df.select(
            join_payment_schedule_df.PaymentScheduleId,
            lit("qca").cast("String").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            join_payment_schedule_df.PaymentScheduleKey.cast("String"),
            dim_cases_static.CASE_ACCOUNT_FOR_SEARCH.cast("String").alias("ExposureKey"),
            fs_cases_snap.cust_code.cast("String").alias("PartyKey"),
            join_payment_schedule_df.AgreementKey.cast("String").alias("AgreementKey"),
            join_payment_schedule_df.sete_code.cast("String").alias("SettlementKey"),
            lit("Settlement Payment Schedule").cast("String").alias("PaymentScheduleTypeCode"),
            lit("Settlement Payment Schedule").cast("String").alias("PaymentScheduleType"),
            join_payment_schedule_df.InstallmentNumber.cast("Int").alias("InstallmentNumber"),
            join_payment_schedule_df.pd_desc.cast("timestamp").alias("InstallmentDate"),
            lit(None).cast("Int").alias("RequestSN"),
            join_payment_schedule_df.seps_installment_type_code.cast("String").alias(
                "InstallmentTypeCode"
            ),
            join_payment_schedule_df.InstallmentType.cast("String").alias(
                "InstallmentType"
            ),
            join_payment_schedule_df.inst_id.cast("String").alias(
                "InstallmentStatusCode"
            ),
            join_payment_schedule_df.InstallmentStatus.cast("String").alias("InstallmentStatus"),
            join_payment_schedule_df.seps_installment_amt.cast("decimal(38,7)").alias(
                "InstallmentAmount"
            ),
            lit(None).cast("decimal(19,2)").alias("InterestAmount"),
            lit(None).cast("decimal(19,2)").alias("CapitalAmount"),
            lit(None).cast("decimal(19,2)").alias("CommissionAmount"),
            lit(None).cast("decimal(19,2)").alias("ExpenseAmount"),
            lit(None).cast("int").alias("OverdueDays"),
            lit(None).cast("decimal(19,2)").alias("InstallmentAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("InterestAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("CapitalAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("CommissionAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("ExpenseAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("RemainingAmount"),
            join_payment_schedule_df.seps_is_down_payment.cast("boolean").alias(
                "DownPaymentFlag"
            ),
            join_payment_schedule_df.seps_code.cast("int").alias("RecordKey"),
            join_payment_schedule_df.setc_code.cast("int").alias(
                "SettlementCaseRecordKey"
            ),
            lit(None).cast("int").alias("PayTolerance"),
            lit(None).cast("decimal(12,6)").alias("PayPercent"),
            lit(None).cast("decimal(18,2)").alias("PayPercentAmount"),
            lit(None).cast("boolean").alias("ImplementationFlag"),
            fs_arrangements_plan.businessdate.cast("date").alias("BusinessDate"),
        )


        # ================== Append ==================

        payment_schedule_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_payment_schedule_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)